## Original Test Case

In [ ]:
!pip install taegis-magic

In [ ]:
%reload_ext taegis_magic

In [ ]:
%taegis auth login --use-universal-authentication

In [ ]:
%%taegis events search --tenant 11063 --assign process_events
from process
WHERE process_correlation_id in ('302c8cb3-7ae4-5264-907e-6716ce554652:25614222880669775:1774877809989093', '302c8cb3-7ae4-5264-907e-6716ce554652:25614222880671395:1774896303136505')
EARLIEST = -30d

In [ ]:
from taegis_magic.pandas.process import process_correlate_netflow

region="charlie"
tenant_id="11063"

In [ ]:
ret_valid_data = process_events.pipe(process_correlate_netflow, region=region, tenant_id=tenant_id, earliest='30d')

## Elite Testing

Grab a few hosts with netflows that are mapped to processes

In [ ]:
%%taegis --no-sdk-warning events search --region charlie --tenant 11772 --assign test_netflow_data
FROM netflow 
WHERE processcorrelationid.pid IS NOT NULL AND processcorrelationid.pid != '0'
EARLIEST = -1d 
| head 5

In [ ]:
from taegis_magic.core.grid import data_grid
from taegis_magic.pandas.utils import groupby

columns = ['hostname', 'host_id', 'processcorrelationid.pid', 'resource_id', 'tenant_id']
sample_netflow_data_grid = groupby(test_netflow_data, columns)
test_netflow_widget = data_grid(sample_netflow_data_grid)
display(test_netflow_widget)

In [ ]:
sample_netflow_data_grid.shape

In [ ]:
sample_netflow_data_grid.columns

<br>

Grab processes noted above

In [ ]:
%%taegis --no-sdk-warning events search --region charlie --tenant 11772 --assign test_data
FROM process 
WHERE host_id IN ('b3c82df29a49c313609cf9c7c7a7af337eeb7b5b', '35f963f88c2b941eaf6d7aa6ced26736', '35f963f8e6cd18488b3c756ec5120b80')
and process_id IN ('8940', '3336', '8600')
EARLIEST = -7d 
| head 5

In [ ]:
from taegis_magic.core.grid import data_grid
from taegis_magic.pandas.utils import groupby

columns = ['hostname', 'username', 'parent_image_path', 'commandline', 'host_id', 'process_correlation_id', 'resource_id', 'tenant_id']
sample_data_grid = groupby(test_data, columns)
test_widget = data_grid(sample_data_grid)
display(test_widget)

*Note results appear*

<br>

Attempt to pull by process_correlation_id

In [ ]:
%%taegis --no-sdk-warning events search --assign process_events --tenant 11772 --region charlie
from process
WHERE process_correlation_id in (
    'b3c82df29a49c313609cf9c7c7a7af337eeb7b5b:8940:1775006876829380', 
    'b3c82df29a49c313609cf9c7c7a7af337eeb7b5b:8600:1774774224527431', 
    'b3c82df29a49c313609cf9c7c7a7af337eeb7b5b:3336:1774652207907826', 
    'b3c82df29a49c313609cf9c7c7a7af337eeb7b5b:3336:1774976054894184',
    'b3c82df29a49c313609cf9c7c7a7af337eeb7b5b:8600:1774821522090907')
EARLIEST = -7d

In [ ]:
from taegis_magic.core.grid import data_grid
from taegis_magic.pandas.utils import groupby

columns = ['hostname', 'username', 'parent_image_path', 'commandline', 'host_id', 'process_correlation_id', 'resource_id', 'tenant_id']
sample_process_events_grid = groupby(process_events, columns)
test_process_events_widget = data_grid(sample_process_events_grid)
display(test_process_events_widget)

In [ ]:
from taegis_magic.pandas.process import process_correlate_netflow

ret_valid_data = test_process_events_widget.get_visible_data().pipe(process_correlate_netflow, region='charlie', tenant_id='11772', earliest='7d')

In [ ]:
data_grid(ret_valid_data)

In [ ]:
ret_valid_data.columns